# Karibu Kenya - AI Tours Concierge
> Your personal AI guide to Kenya — safaris, beaches, culture, itineraries and more
> Powered by **Anthropic Claude** | **LangChain** | **Gradio**

---
### What this concierge does:
- Answers questions about safaris, wildlife, national parks
- Recommends beaches, coastal towns, and islands
- Builds custom itineraries based on your dates and interests
- Suggests hotels and lodges across all budget tiers
- Gives practical travel advice (visas, transport, health, money)
- Searches the web for live prices, reviews, and current conditions
- Maintains full conversation memory across your chat session

## Cell 1 - Install Dependencies

In [ ]:
import subprocess, sys

packages = [
    "anthropic>=0.40.0",
    "langchain>=0.3.0",
    "langchain-anthropic>=0.3.0",
    "langchain-community>=0.3.0",
    "langchain-core>=0.3.0",
    "gradio>=4.0",
    "python-dotenv",
    "requests",
    "duckduckgo-search>=6.0",
]

for pkg in packages:
    print(f"Installing {pkg}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  WARNING: {result.stderr.strip()[:120]}")

print("\nAll packages installed.")

## Cell 2 - Configuration and API Keys (Anthropic + Ollama)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
ANTHROPIC_MODEL   = "claude-sonnet-4-6"

OLLAMA_MODEL    = os.getenv("OLLAMA_MODEL",    "llama3.2")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

# ============================================================
# PROVIDER SELECTION
# ============================================================
# 'auto'      -> try Anthropic first, fall back to Ollama on any error
# 'anthropic' -> Anthropic only
# 'ollama'    -> Ollama only
LLM_PROVIDER = "auto"


if ANTHROPIC_API_KEY:
    print(f"Anthropic : key loaded ({ANTHROPIC_API_KEY[:12]}...)")
else:
    print("Anthropic : no API key found")
print(f"Ollama    : {OLLAMA_BASE_URL}  model={OLLAMA_MODEL}")
print(f"Provider  : {LLM_PROVIDER}")
print("=" * 50)


## Cell 3 - Kenya Concierge System Prompt

In [ ]:
KENYA_SYSTEM_PROMPT = """
You are Karibu (Welcome in Swahili), an expert Kenya tourism concierge with deep,
encyclopedic knowledge of Kenya's destinations, wildlife, culture, and travel logistics.
You have the warm personality of a knowledgeable local guide deeply passionate about Kenya.

=== YOUR DEEP EXPERTISE ===

WILDLIFE AND SAFARIS:
- Masai Mara National Reserve: Great Migration (Jul-Oct), Big Five year-round, balloon safaris
- Amboseli National Park: Elephant herds, Kilimanjaro backdrop, swamps and bush walks
- Tsavo East and West: Red elephants, Mzima Springs, Lugard Falls, vast wilderness
- Samburu National Reserve: Rare Northern Species (Grevy's zebra, reticulated giraffe, Beisa oryx)
- Lake Nakuru NP: Flamingos, white/black rhinos, Rothschild giraffes, leopards
- Ol Pejeta Conservancy: Last northern white rhinos, chimpanzee sanctuary, Big Five
- Laikipia Plateau: Wilderness retreats, camel safaris, community conservancies
- Hell's Gate NP: Cycling, rock climbing, geothermal, Maasai gorge walks
- Lake Bogoria: Flamingos and hot springs in the Rift Valley
- Chyulu Hills: Green Hills of Africa, lava tubes, hidden springs

COAST AND BEACHES:
- Diani Beach: Top-rated white sand beach, coral reef snorkeling, water sports, colobus monkeys
- Lamu Island: UNESCO World Heritage Site, ancient Swahili culture, dhow sailing, car-free town
- Malindi: Italian expat community, Malindi Marine NP, Vasco da Gama pillar, kite surfing
- Watamu: Marine protected area, sea turtle conservation, Arabuko-Sokoke forest
- Mombasa Old Town: Fort Jesus, Swahili architecture, spice markets, dhow harbour
- Kilifi: Creek lifestyle, Boho Vipingo Ridge, kitesurfing mecca
- Funzi Island: Remote mangrove island, exclusive eco-lodge, kayaking

MOUNTAINS AND HIKING:
- Mount Kenya (5,199m): Africa's second highest, multiple routes (Sirimon, Naro Moru, Chogoria)
- Aberdare Ranges: Tree hotel stays (Treetops, The Ark), moorland hikes, waterfalls
- Karura Forest Nairobi: Urban forest biking and walking trails
- Ngong Hills: Easy day hike from Nairobi with panoramic Rift Valley views
- Mount Longonot: Crater rim hike in the Rift Valley

NAIROBI HIGHLIGHTS:
- Giraffe Centre: Hand-feed endangered Rothschild giraffes
- David Sheldrick Elephant Orphanage: Baby elephant rescue programme
- Nairobi National Park: Only capital city with a national park
- Karen Blixen Museum: Out of Africa author's former home
- Bomas of Kenya: Traditional cultural performances
- Kazuri Beads: Women-owned bead factory and social enterprise
- Carnivore Restaurant: Famous nyama choma (grilled meat) institution

CULTURAL EXPERIENCES:
- Maasai villages (Manyattas) near Masai Mara and Amboseli — beading, dances, warrior culture
- Samburu community visits in Northern Kenya
- Swahili culture immersion in Lamu — taarab music, Eid festivals, dhow boat building
- Kikuyu farming culture in the Central Highlands

ACCOMMODATION TIERS:
Budget (under $100/night): Wildebeest Eco Camp Nairobi, Milele Beach Hotel Diani, guesthouses in Lamu
Mid-range ($100-300/night): Keekorok Lodge Mara, Ol Tukai Lodge Amboseli, Pinewood Beach Diani
Luxury ($300-800/night): Sarova Mara Game Camp, Tortilis Camp Amboseli, Alfajiri Villas Diani
Ultra-luxury ($800+/night): andBeyond Bateleur Camp, Singita Mara River Tented Camp,
                             Sasaab Samburu, Sirikoi Lewa, Galdessa Tsavo

PRACTICAL TRAVEL INFO:
- Visas: e-Visa required for most nationalities (apply at evisa.go.ke), $51 USD
- Currency: Kenyan Shilling (KES), approx 130 KES to $1 USD. USD widely accepted in tourism areas
- Health: Yellow fever vaccination required from endemic countries. Malaria prophylaxis recommended.
  Travel insurance essential
- Internal flights: Safarilink, AirKenya, Fly540 connect Nairobi to Mara, Amboseli, Coast, Samburu
- Roads: Nairobi-Mombasa SGR train (4.5hrs, very comfortable). Roads to parks vary, 4WD often needed
- Tipping: 10-15% in restaurants. Safari guides: $10-20/day. Lodge staff: $5-10/day per person
- Best overall time: July-October (dry season, Great Migration). Second best: January-February

SEASONAL GUIDE:
- Jan-Feb: Warm and dry, great for Amboseli, coast beaches, calving season in Mara
- Mar-May: Long rains, lower prices, lush landscapes, some roads impassable
- Jun-Jul: Migration builds in Mara, perfect weather, slightly cheaper than peak
- Jul-Oct: PEAK SEASON — Great Migration crossings, best wildlife viewing, dry and sunny
- Nov: Short rains begin, birdwatching excellent, migratory birds arrive
- Dec: Festive season, coast becomes busy, Mara still excellent

HIDDEN GEMS AND INSIDER TIPS:
- Sirikoi Lodge in Lewa: Most exclusive private conservancy experience
- Hell's Gate by bicycle: Rent bikes at the gate, cycle among zebras and buffaloes
- Kitich Forest Camp: Remote fly-in camp in Matthews Range, almost zero tourists
- Funzi Island: Canoe through mangroves, completely off the tourist trail
- Treetops and The Ark: Spend a night watching animals at a floodlit waterhole
- Nairobi food scene: Talisman, Cultiva, Mama Oliech (fresh fish), Carnivore

=== RESPONSE STYLE ===
- Be warm, enthusiastic, and conversational like a knowledgeable Kenyan friend
- Use vivid, sensory language that makes Kenya come alive
- Sprinkle in Swahili words with translations: Hakuna matata (no worries)!
- For itineraries: give specific day-by-day plans with timing
- Always mention BEST TIME to visit and INSIDER TIP for each destination
- Recommend across 3 budget tiers when suggesting accommodation
- Keep responses structured with short paragraphs
- Always end with an invitation to ask follow-up questions
"""

print("System prompt configured.")
print(f"Prompt length: {len(KENYA_SYSTEM_PROMPT)} characters")

## Cell 4 - LLM Setup: Anthropic (primary) + Ollama (fallback)

- **Anthropic** is used when a valid `ANTHROPIC_API_KEY` is set.
- **Ollama** is used as a local fallback when Anthropic fails or has no credits.
- Set `LLM_PROVIDER = 'ollama'` to force local mode, or `'anthropic'` to disable fallback.
- Ollama must be running locally: `ollama serve` + `ollama pull llama3.2`

In [ ]:

import requests as _req
from langchain_anthropic import ChatAnthropic
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool

def _ollama_available() -> bool:
    try:
        r = _req.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=3)
        return r.status_code == 200
    except Exception:
        return False

def _anthropic_available() -> bool:
    return bool(ANTHROPIC_API_KEY and ANTHROPIC_API_KEY.startswith("sk-ant"))

anthropic_llm = None
ollama_llm    = None

if _anthropic_available():
    anthropic_llm = ChatAnthropic(
        model=ANTHROPIC_MODEL,
        anthropic_api_key=ANTHROPIC_API_KEY,
        temperature=0.7,
        max_tokens=1500,
    )
    print(f"Anthropic LLM ready  : {ANTHROPIC_MODEL}")
else:
    print("Anthropic LLM        : skipped (no valid key)")

if _ollama_available():
    ollama_llm = ChatOllama(
        model=OLLAMA_MODEL,
        base_url=OLLAMA_BASE_URL,
        temperature=0.7,
    )
    print(f"Ollama LLM ready     : {OLLAMA_MODEL} @ {OLLAMA_BASE_URL}")
else:
    print(f"Ollama LLM           : not reachable at {OLLAMA_BASE_URL}")

if not anthropic_llm and not ollama_llm:
    print("WARNING: No LLM available. Set ANTHROPIC_API_KEY or start Ollama.")

_search = DuckDuckGoSearchRun()

@tool
def search_kenya_info(query: str) -> str:
    """Search the web for up-to-date Kenya tourism info."""
    try:
        return _search.run(f"Kenya tourism {query} 2025") or "No results found."
    except Exception as e:
        return f"Search unavailable: {e}"

@tool
def get_kenya_weather(region: str) -> str:
    """Get current weather and seasonal travel info for a Kenya region."""
    try:
        return _search.run(f"current weather {region} Kenya travel conditions 2025")
    except Exception as e:
        return f"Weather data unavailable: {e}"

@tool
def search_kenya_hotels(location: str, budget: str = "any") -> str:
    """Search for hotels, lodges, and camps in a Kenya location."""
    try:
        return _search.run(f"best {budget} hotels lodges {location} Kenya prices reviews 2025")
    except Exception as e:
        return f"Hotel search unavailable: {e}"

@tool
def search_kenya_activities(location: str, activity_type: str = "") -> str:
    """Search for activities, tours, and experiences in a Kenya destination."""
    try:
        return _search.run(f"{activity_type} activities things to do {location} Kenya 2025")
    except Exception as e:
        return f"Activity search unavailable: {e}"

TOOLS   = [search_kenya_info, get_kenya_weather, search_kenya_hotels, search_kenya_activities]
TOOL_MAP = {t.name: t for t in TOOLS}

anthropic_llm_tools = anthropic_llm.bind_tools(TOOLS) if anthropic_llm else None

def chat_with_llm(message: str):
    """
    Route the user message through the tool-enabled Anthropic LLM if available,
    else fall back to plain Ollama chat.
    """
    if anthropic_llm_tools:
        try:
            return anthropic_llm_tools.invoke({"input": message}).get("output")
        except Exception as e:
            return f"Anthropic error: {e}"
    elif ollama_llm:
        try:
            return ollama_llm.chat([HumanMessage(content=message)]).content
        except Exception as e:
            return f"Ollama error: {e}"
    else:
        return "No LLM available. Please configure ANTHROPIC_API_KEY or start Ollama."

print(f"\nTools registered: {list(TOOL_MAP.keys())}")
print("Dual-provider LLM setup complete.")

## Cell 5 - Chat Function with Automatic Fallback

Tries Anthropic first. If it raises a credit, auth, or network error,
automatically retries with Ollama and appends a small notice to the reply.

In [ ]:

_ANTHROPIC_CREDIT_ERRORS = (
    "credit balance",
    "insufficient_quota",
    "rate_limit",
    "overloaded",
    "529",
    "402",
    "401",
    "authentication",
    "permission",
    "billing",
)

def _is_credit_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    return any(kw in msg for kw in _ANTHROPIC_CREDIT_ERRORS)

def _run_tool_loop(llm_with_tools, messages: list, max_rounds: int = 4) -> str:
    """Run tool-use loop for a tool-enabled LLM (Anthropic only)."""
    for _ in range(max_rounds):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            break

        for tc in response.tool_calls:
            tool_fn = TOOL_MAP.get(tc["name"])
            try:
                result = tool_fn.invoke(tc["args"]) if tool_fn else f"Unknown tool: {tc['name']}"
            except Exception as e:
                result = f"Tool error: {e}"
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    reply = response.content
    if isinstance(reply, list):
        reply = " ".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in reply
        )
    return str(reply).strip() or "Samahani, I could not generate a response."

def _build_messages(user_message: str, history: list) -> list:
    messages = [SystemMessage(content=KENYA_SYSTEM_PROMPT)]
    for pair in history:
        human = pair[0] if len(pair) > 0 else None
        ai    = pair[1] if len(pair) > 1 else None
        if human:
            messages.append(HumanMessage(content=human))
        if ai:
            messages.append(AIMessage(content=ai))
    messages.append(HumanMessage(content=user_message))
    return messages

def chat_with_karibu(user_message: str, history: list) -> tuple:
    """Gradio chat handler: Anthropic first, Ollama fallback (plain chat)."""
    if not user_message.strip():
        return "", history

    if not anthropic_llm_tools and not ollama_llm:
        msg = (
            "No LLM is available.\n"
            "- For Anthropic: set ANTHROPIC_API_KEY in Cell 2.\n"
            "- For Ollama: install from https://ollama.com and run `ollama pull llama3.2`."
        )
        history.append([user_message, msg])
        return "", history

    provider_used = None
    reply         = None
    fallback_note = ""

    if LLM_PROVIDER in ("auto", "anthropic") and anthropic_llm_tools:
        try:
            messages = _build_messages(user_message, history)
            reply    = _run_tool_loop(anthropic_llm_tools, messages)
            provider_used = "Anthropic"
        except Exception as e:
            err = str(e)
            print(f"[Anthropic error] {err[:200]}")
            if LLM_PROVIDER == "anthropic":
                history.append([user_message, f"Anthropic error: {err[:300]}"])
                return "", history
            fallback_note = (
                "\n\n*(Anthropic unavailable — insufficient credits or error. Using Ollama locally.)*"
                if _is_credit_error(e) else
                f"\n\n*(Anthropic error: {err[:120]}. Switched to Ollama.)*"
            )

    if reply is None and LLM_PROVIDER in ("auto", "ollama") and ollama_llm:
        try:
            messages = _build_messages(user_message, history)
            response = ollama_llm.invoke(messages)
            reply = response.content if isinstance(response.content, str) else str(response.content)
            provider_used = f"Ollama ({OLLAMA_MODEL})"
        except Exception as e:
            err = str(e)
            print(f"[Ollama error] {err[:200]}")
            history.append([
                user_message,
                f"Both providers failed.\n\nAnthropic: {fallback_note}\nOllama: {err[:200]}"
            ])
            return "", history

    if reply is None:
        reply = "No LLM provider was able to respond. Check your configuration."

    print(f"[provider] {provider_used}")
    history.append([user_message, reply + fallback_note])
    return "", history

print("Chat function ready.")
print(f"Active provider mode : {LLM_PROVIDER}")
print(f"Anthropic available  : {anthropic_llm_tools is not None}")
print(f"Ollama available     : {ollama_llm is not None}")

## Cell 6 - Gradio UI and Launch

In [ ]:
import gradio as gr

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@600;700&family=Nunito:wght@300;400;500;600&display=swap');

body, .gradio-container {
    background: linear-gradient(135deg, #0d1f13 0%, #1a2e1e 50%, #0a1508 100%) !important;
    font-family: 'Nunito', sans-serif !important;
}

.main-title {
    font-family: 'Playfair Display', serif !important;
    font-size: 2.2em !important;
    font-weight: 700 !important;
    background: linear-gradient(135deg, #c8a96e, #f0d5a0, #c8a96e) !important;
    -webkit-background-clip: text !important;
    -webkit-text-fill-color: transparent !important;
    background-clip: text !important;
    text-align: center;
    padding-top: 16px;
}

.subtitle {
    color: rgba(240,235,224,0.6) !important;
    font-size: 0.95em !important;
    text-align: center;
    margin-bottom: 8px;
}

#karibu-chat {
    border: 1px solid rgba(200,169,110,0.25) !important;
    border-radius: 14px !important;
    background: rgba(255,255,255,0.03) !important;
}

#msg-input textarea {
    background: rgba(255,255,255,0.06) !important;
    border: 1px solid rgba(200,169,110,0.3) !important;
    border-radius: 10px !important;
    color: #f0ebe0 !important;
    font-family: 'Nunito', sans-serif !important;
    font-size: 14px !important;
}

#msg-input textarea:focus {
    border-color: #c8a96e !important;
    box-shadow: 0 0 0 2px rgba(200,169,110,0.2) !important;
}

#send-btn {
    background: linear-gradient(135deg, #c8a96e, #8b6914) !important;
    border: none !important;
    border-radius: 10px !important;
    color: white !important;
    font-weight: 600 !important;
}

#send-btn:hover {
    box-shadow: 0 4px 14px rgba(200,169,110,0.4) !important;
}

#clear-btn {
    background: rgba(255,255,255,0.05) !important;
    border: 1px solid rgba(255,255,255,0.15) !important;
    color: rgba(240,235,224,0.7) !important;
    border-radius: 10px !important;
}

.quick-btn {
    background: rgba(255,255,255,0.05) !important;
    border: 1px solid rgba(200,169,110,0.2) !important;
    color: rgba(240,235,224,0.8) !important;
    border-radius: 20px !important;
    font-size: 13px !important;
    transition: all 0.2s !important;
}

.quick-btn:hover {
    background: rgba(200,169,110,0.15) !important;
    border-color: rgba(200,169,110,0.5) !important;
    color: #c8a96e !important;
}

.region-btn {
    background: rgba(45,106,79,0.2) !important;
    border: 1px solid rgba(200,169,110,0.25) !important;
    color: rgba(240,235,224,0.85) !important;
    border-radius: 8px !important;
    font-size: 13px !important;
    transition: all 0.2s !important;
}

.region-btn:hover {
    background: rgba(200,169,110,0.2) !important;
    border-color: #c8a96e !important;
    color: #c8a96e !important;
}

.section-label {
    color: rgba(200,169,110,0.8) !important;
    font-size: 11px !important;
    font-weight: 600 !important;
    letter-spacing: 1.5px !important;
    text-transform: uppercase !important;
}

.footer-text {
    color: rgba(240,235,224,0.28) !important;
    text-align: center !important;
    font-size: 11px !important;
    padding-bottom: 10px;
}
"""

QUICK_PROMPTS = [
    "Best safari for the Great Migration?",
    "Top beaches near Mombasa?",
    "Plan a 10-day Kenya itinerary",
    "Best time to visit Amboseli?",
    "Climbing Mount Kenya — what to know?",
    "Budget safari options under $200 per day?",
    "Cultural experiences in Lamu Island",
    "Family-friendly Kenya activities?",
]

KENYA_REGIONS = [
    "Nairobi",
    "Masai Mara",
    "Amboseli",
    "Diani Beach",
    "Lamu Island",
    "Samburu",
    "Watamu",
    "Ol Pejeta",
]

WELCOME_MESSAGE = (
    "**Karibu sana! Welcome to Kenya — the jewel of East Africa!**\n\n"
    "I am Karibu, your personal travel concierge. I can help you plan the perfect "
    "Kenyan adventure — from witnessing the Great Migration in the Masai Mara "
    "to relaxing on Diani's powder-white beaches or trekking up Mount Kenya.\n\n"
    "I have live web search to find you current hotel prices, park conditions, "
    "and real traveler reviews.\n\n"
    "Tell me your **travel dates, interests, and budget** — and let's start "
    "planning your dream Kenya trip!"
)

with gr.Blocks(
    css=CUSTOM_CSS,
    title="Karibu Kenya — Tourism Concierge",
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.emerald,
        neutral_hue=gr.themes.colors.slate,
        font=[gr.themes.GoogleFont("Nunito"), "ui-sans-serif", "sans-serif"],
    )
) as demo:

    gr.Markdown("<div class='main-title'>Karibu Kenya</div>")
    gr.Markdown(
        "<div class='subtitle'>AI-Powered Tourism Concierge "
        "&mdash; Safaris &middot; Beaches &middot; Culture &middot; Travel Planning</div>"
    )

    gr.Markdown("<div class='section-label'>Explore by Region</div>")
    with gr.Row():
        region_btns = []
        for region in KENYA_REGIONS:
            btn = gr.Button(region, size="sm", elem_classes="region-btn")
            region_btns.append((btn, region))

    chatbot = gr.Chatbot(
        value=[[None, WELCOME_MESSAGE]],
        height=460,
        elem_id="karibu-chat",
        label="",
        show_label=False,
        bubble_full_width=False,
        show_copy_button=True,
        render_markdown=True,
    )

    with gr.Row():
        msg_input = gr.Textbox(
            placeholder="Ask about safaris, beaches, hotels, itineraries, visas, what to pack...",
            show_label=False,
            scale=8,
            lines=1,
            max_lines=4,
            elem_id="msg-input"
        )
        send_btn = gr.Button("Send", scale=1, variant="primary", elem_id="send-btn")
        clear_btn = gr.Button("Clear", scale=1, elem_id="clear-btn")

    gr.Markdown("<div class='section-label'>Quick Questions</div>")
    with gr.Row():
        quick_btns = []
        for prompt_text in QUICK_PROMPTS[:4]:
            qb = gr.Button(prompt_text, size="sm", elem_classes="quick-btn")
            quick_btns.append((qb, prompt_text))
    with gr.Row():
        for prompt_text in QUICK_PROMPTS[4:]:
            qb = gr.Button(prompt_text, size="sm", elem_classes="quick-btn")
            quick_btns.append((qb, prompt_text))

    with gr.Accordion("About this Concierge", open=False):
        gr.Markdown("""
        ### Technology Stack
        - **AI Engine**: Anthropic Claude via LangChain
        - **Memory**: Full conversation history preserved within the session
        - **Web Search**: DuckDuckGo live search for current hotels, prices, reviews, and conditions
        - **Tools**: Kenya hotel search, activity search, weather lookup, general info search
        - **UI**: Gradio with custom Kenya-themed design

        ### What Karibu Knows
        - All major national parks and conservancies
        - Coastal destinations from Mombasa to Lamu
        - Accommodation across budget to ultra-luxury
        - Seasonal guides and best times to visit each region
        - Internal flights, ground transport, and road conditions
        - Visa requirements, health advice, currency, and tipping norms
        - Cultural experiences, food, and hidden gems

        ### Setup
        Set ANTHROPIC_API_KEY in Cell 2 or in a .env file.
        Requires Python 3.9+ with the packages installed in Cell 1.

        Msafiri salama — Safe travels!
        """)

    gr.Markdown(
        "<div class='footer-text'>"
        "Powered by Anthropic Claude &middot; LangChain &middot; Gradio"
        "&nbsp;&mdash;&nbsp; Msafiri salama"
        "</div>"
    )

    # Wire up send / enter
    send_btn.click(
        fn=chat_with_karibu,
        inputs=[msg_input, chatbot],
        outputs=[msg_input, chatbot]
    )
    msg_input.submit(
        fn=chat_with_karibu,
        inputs=[msg_input, chatbot],
        outputs=[msg_input, chatbot]
    )

    # Clear resets to the welcome message
    clear_btn.click(
        fn=lambda: ([[None, WELCOME_MESSAGE]], ""),
        outputs=[chatbot, msg_input]
    )

    # Region buttons
    for btn, region in region_btns:
        btn.click(
            fn=lambda hist, r=region: chat_with_karibu(
                f"Tell me everything about visiting {r}, Kenya — "
                f"best time, top experiences, where to stay, and insider tips.",
                hist
            ),
            inputs=[chatbot],
            outputs=[msg_input, chatbot]
        )

    # Quick prompt buttons
    for qbtn, qtext in quick_btns:
        qbtn.click(
            fn=lambda hist, t=qtext: chat_with_karibu(t, hist),
            inputs=[chatbot],
            outputs=[msg_input, chatbot]
        )

print("Launching Karibu Kenya Concierge...")
print("Opening at http://localhost:7860")
print("Set share=True for a public Gradio link.")
print("Press Ctrl+C to stop the server.")

demo.launch(
    inbrowser=True,
    share=False,
    server_name="0.0.0.0",
    server_port=7860,
    show_error=True,
)